[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyneuro/SCP/blob/main/extra_notebooks/act_segmentation.ipynb)

# Optional Notebook - ACT Channel Segmentation

This notebook is the cleaned SCP version of the old `1_segment.ipynb` helper. It is **not** part of the main numbered SCP pipeline. Use it when you want ACT-style channel segmentation support before passive/active tuning.

Workflow:
- **S.0 Environment Setup**: configure local/Colab imports and locate SCP + ACT.
- **S.1 Select Segmentation Parameters**: set voltage grid, cutoffs, and channel-module groupings.
- **S.2 Define Activation Curves**: enter/adjust steady-state activation equations from modfiles.
- **S.3 Run ACT Segregator**: generate segmented activation curves and printed modfile-edit instructions.
- **S.4 Plot Segmented Curves**: compare segmented curves by functional module.
- **S.5 Apply Segmentation Manually**: copy/edit tune `modfiles/` based on ACT output.
- **S.6 Optional Channel Curve Checks**: quick activation/inactivation sanity plots.

Typical use: validate a raw `orig` tune, copy it to `tuned`, run this notebook to determine any needed segmentation edits, manually update the copied modfiles, then continue with `2_passive.ipynb` and `3_active.ipynb` using `tuned`.


## S.0 Environment Setup

Run this first. Locally it finds your SCP repo and sibling ACT checkout. In Colab it can clone SCP and ACT automatically.


In [ ]:
# Environment setup: works locally or in Google Colab
%load_ext autoreload
%autoreload 2

import os
import subprocess
import sys
from pathlib import Path

# User-editable only when running in a fresh Colab or unusual local layout.
SCP_REPO_URL = os.environ.get("SCP_REPO_URL", "https://github.com/cyneuro/SCP.git")
SCP_REPO_BRANCH = os.environ.get("SCP_REPO_BRANCH", "") or None
SCP_REPO_DIR = Path(os.environ.get("SCP_REPO_DIR", "/content/SCP" if "COLAB_RELEASE_TAG" in os.environ else str(Path.cwd() / "SCP")))

ACT_REPO_URL = os.environ.get("ACT_REPO_URL", "https://github.com/V-Marco/ACT.git")
ACT_REPO_BRANCH = os.environ.get("ACT_REPO_BRANCH", "") or None
ACT_REPO_DIR = Path(os.environ.get("ACT_REPO_DIR", "/content/mods/ACT" if "COLAB_RELEASE_TAG" in os.environ else str(Path.cwd().parent / "mods" / "ACT")))

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
AUTO_CLONE_REPO = os.environ.get("SCP_AUTO_CLONE", "1" if IN_COLAB else "0") not in {"0", "false", "False"}
AUTO_CLONE_ACT = os.environ.get("SCP_AUTO_CLONE_ACT", "1" if IN_COLAB else "0") not in {"0", "false", "False"}
INSTALL_DEPS = None  # None = install automatically in Colab, do not install locally.


def _looks_like_scp_repo(path: Path) -> bool:
    return (path / "modules").is_dir() and (path / "run_pipeline.py").is_file()


def _clone_repo(url: str, dest: Path, branch: str | None = None) -> Path:
    dest = dest.expanduser()
    if dest.exists():
        return dest.resolve()
    dest.parent.mkdir(parents=True, exist_ok=True)
    clone_url = url
    token = os.environ.get("SCP_GIT_TOKEN") or os.environ.get("SCP_GITHUB_TOKEN") or os.environ.get("GITHUB_TOKEN")
    if token and clone_url.startswith("https://") and "@" not in clone_url:
        clone_url = clone_url.replace("https://", f"https://{token}@", 1)
    cmd = ["git", "clone", "--depth", "1"]
    if branch:
        cmd += ["--branch", branch]
    cmd += [clone_url, str(dest)]
    subprocess.check_call(cmd)
    return dest.resolve()


# Minimal pre-import bootstrap. Fresh Colab cannot import SCP helpers until the repo exists.
repo_root = None
env_root = os.environ.get("SCP_ROOT")
if env_root and _looks_like_scp_repo(Path(env_root).expanduser()):
    repo_root = Path(env_root).expanduser().resolve()
else:
    start = Path.cwd().resolve()
    candidates = list((start, *start.parents))
    for base in (start, start.parent):
        try:
            candidates.extend(child for child in base.iterdir() if child.is_dir())
        except Exception:
            pass
    for candidate in candidates:
        if _looks_like_scp_repo(candidate):
            repo_root = candidate.resolve()
            break

if repo_root is None:
    if not AUTO_CLONE_REPO:
        raise FileNotFoundError("Could not find SCP. Set SCP_ROOT or enable SCP_AUTO_CLONE=1.")
    repo_root = _clone_repo(SCP_REPO_URL, SCP_REPO_DIR, SCP_REPO_BRANCH)

os.environ["SCP_ROOT"] = str(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from modules.notebooks.bootstrap import ensure_notebook_dependencies
from modules.notebooks.helpers import ensure_external_repo_on_syspath

ensure_notebook_dependencies(install_deps=INSTALL_DEPS)

try:
    act_path = ensure_external_repo_on_syspath(
        repo_name="ACT",
        marker_rel=Path("act") / "segregation.py",
        env_vars=("SCP_ACT_PATH", "ACT_PATH", "ACT_ROOT"),
        repo_root=repo_root,
        prepend=False,
    )
except FileNotFoundError:
    if not AUTO_CLONE_ACT:
        raise
    act_path = _clone_repo(ACT_REPO_URL, ACT_REPO_DIR, ACT_REPO_BRANCH)
    if str(act_path) not in sys.path:
        sys.path.append(str(act_path))

import matplotlib.pyplot as plt
import numpy as np
from act.segregation import ACTSegregator

print("Runtime:", "Colab" if IN_COLAB else "local")
print("SCP repo:", repo_root)
print("ACT repo:", act_path)


## S.1 Select Segmentation Parameters

Adjust these values for the model being segmented. The channel names must match the activation-function dictionary in the next section.


In [ ]:
# Voltage grid used to evaluate activation curves.
V_MIN_MV = -150.0
V_MAX_MV = 80.0
N_POINTS = 1000

# Defaults are from the previous SCP segmentation helper. Adjust for a new cell/model.
PASSIVE_CUTOFF_MV = -79.66
LTO_CUTOFF_MV = -79.66
SPIKING_CUTOFF_MV = -60.0
BURSTING_CUTOFF_MV = -40.0
EXTRAPOLATE_DV = 2.0

# Functional modules and channel names used by the example equations below.
SEGMENTATION_MODULES = [
    ("Passive", ["H"], PASSIVE_CUTOFF_MV),
    ("LTO", ["Nap", "Imv2", "K_T"], LTO_CUTOFF_MV),
    ("Spiking", ["NaTa", "Kd"], SPIKING_CUTOFF_MV),
    ("Bursting", ["CaLVA", "CaHVA", "Kv2like", "Kv31"], BURSTING_CUTOFF_MV),
]

v = np.linspace(V_MIN_MV, V_MAX_MV, N_POINTS)
print(f"Voltage grid: {V_MIN_MV} to {V_MAX_MV} mV ({N_POINTS} points)")


## S.2 Define Activation Curves

These helper functions provide editable activation-curve definitions for ACT-style segmentation and should be adjusted when segmenting a different model. The plot is a quick sanity check before running ACT segmentation.


In [ ]:
def vtrap(x, y):
    x = np.asarray(x)
    ratio = x / y
    return np.where(np.abs(ratio) < 1e-6, y * (1 - ratio / 2), x / (np.exp(ratio) - 1))


def H(v):
    mAlpha = 0.001 * 6.43 * vtrap(v + 154.9, 11.9)
    mBeta = 0.001 * 193 * np.exp(v / 33.1)
    return mAlpha / (mAlpha + mBeta)


def NaTa(v):
    malphaF = 0.182
    mbetaF = 0.124
    mvhalf = -48
    mk = 6
    mAlpha = malphaF * vtrap(-(v - mvhalf), mk)
    mBeta = mbetaF * vtrap((v - mvhalf), mk)
    return mAlpha / (mAlpha + mBeta)


def Nap(v):
    return 1.0 / (1 + np.exp((v - -52.6) / -4.6))


def Kd(v):
    return 1 - 1 / (1 + np.exp((v - (-43)) / 8))


def Kv2like(v):
    mAlpha = 0.12 * vtrap(-(v - 43), 11.0)
    mBeta = 0.02 * np.exp(-(v + 1.27) / 120)
    return mAlpha / (mAlpha + mBeta)


def Kv31(v):
    return 1 / (1 + np.exp(((v - 18.700) / -9.700)))


def K_T(v):
    return 1 / (1 + np.exp(-(v - (-47)) / 29))


def Imv2(v):
    mAlpha = 3.3e-3 * np.exp(2.5 * 0.04 * (v - -35))
    mBeta = 3.3e-3 * np.exp(-2.5 * 0.04 * (v - -35))
    return mAlpha / (mAlpha + mBeta)


# SK is Ca-dependent, so it is intentionally not included here.
def CaHVA(v):
    mAlpha = 0.055 * vtrap(-27 - v, 3.8)
    mBeta = 0.94 * np.exp((-75 - v) / 17)
    return mAlpha / (mAlpha + mBeta)


def CaLVA(v):
    return 1.0 / (1 + np.exp((v - -30.0) / -6))


ACTIVATION_FUNCTIONS = {
    "H": H,
    "Nap": Nap,
    "Imv2": Imv2,
    "K_T": K_T,
    "NaTa": NaTa,
    "Kd": Kd,
    "CaLVA": CaLVA,
    "CaHVA": CaHVA,
    "Kv2like": Kv2like,
    "Kv31": Kv31,
}

plt.figure(figsize=(9, 5))
for channel_name, func in ACTIVATION_FUNCTIONS.items():
    plt.plot(v, func(v), label=channel_name)
for _, _, cutoff in SEGMENTATION_MODULES:
    plt.axvline(cutoff, color="black", ls="--", alpha=0.25)
plt.title("Original activation curves")
plt.xlabel("Membrane potential (mV)")
plt.ylabel("Activation")
plt.ylim(-0.05, 1.05)
plt.grid(True, alpha=0.25)
plt.legend(ncol=2)
plt.show()


## S.3 Run ACT Segregator

`ACTSegregator.segregate()` prints the modfile edit instructions and returns segmented activation curves. Copy the printed instructions into the appropriate duplicated tune `modfiles/` manually.


In [ ]:
segregator = ACTSegregator()
segregated_curves = {}

for module_name, channel_names, cutoff in SEGMENTATION_MODULES:
    missing = [name for name in channel_names if name not in ACTIVATION_FUNCTIONS]
    if missing:
        raise KeyError(f"Missing activation functions for {module_name}: {missing}")

    print(f"\n=== {module_name} module | cutoff={cutoff} mV ===")
    curves = [ACTIVATION_FUNCTIONS[name](v) for name in channel_names]
    segregated_curves[module_name] = segregator.segregate(
        v=v,
        activation_curves=curves,
        cutoff_v=cutoff,
        extrapolate_dv=EXTRAPOLATE_DV,
    )


## S.4 Plot Segmented Curves

Use this plot to verify that the segmented curves have the expected cutoff behavior before editing modfiles.


In [ ]:
plt.figure(figsize=(9, 5))

for module_name, channel_names, cutoff in SEGMENTATION_MODULES:
    for curve, channel_name in zip(segregated_curves[module_name], channel_names):
        plt.plot(v, curve, label=f"{module_name}: {channel_name}")
    plt.axvline(cutoff, color="black", ls="--", alpha=0.25)

plt.title("Segmented activation curves")
plt.xlabel("Membrane potential (mV)")
plt.ylabel("Segmented activation")
plt.ylim(-0.05, 1.05)
plt.grid(True, alpha=0.25)
plt.legend(ncol=2)
plt.show()


## S.5 Apply Segmentation Manually

Recommended SCP workflow:

1. Validate the raw tune directory, usually `cells/<CELL>/tunes/orig`.
2. Copy `orig` to `cells/<CELL>/tunes/tuned` before editing.
3. Edit the copied `tuned/modfiles/` using the printed ACT instructions from S.3.
4. Recompile mechanisms in the `tuned` tune.
5. Continue with `2_passive.ipynb` and `3_active.ipynb` using `tuned`.

This notebook intentionally does not overwrite modfiles automatically.


## S.6 Optional Channel Curve Checks

Small standalone plots for checking individual channel activation/inactivation equations. These are helper diagnostics, not required for the segmentation workflow.


In [ ]:
def plot_nata_gate_check():
    V = np.linspace(-100, 50, 301)

    malphaF = 0.182
    mbetaF = 0.124
    mvhalf = -48.0
    mk = 6.0

    halphaF = 0.015
    hbetaF = 0.015
    hvhalf = -69.0
    hk = 6.0

    mAlpha = malphaF * vtrap(-(V - mvhalf), mk)
    mBeta = mbetaF * vtrap((V - mvhalf), mk)
    mInf = mAlpha / (mAlpha + mBeta)

    hAlpha = halphaF * vtrap(V - hvhalf, hk)
    hBeta = hbetaF * vtrap(-(V - hvhalf), hk)
    hInf = hAlpha / (hAlpha + hBeta)

    plt.figure(figsize=(8, 5))
    plt.plot(V, mInf, label="m∞ activation", linewidth=2)
    plt.plot(V, hInf, label="h∞ inactivation", linewidth=2)
    plt.xlabel("Membrane potential (mV)")
    plt.ylabel("Steady-state value")
    plt.title("NaTa Activation and Inactivation Curves")
    plt.legend()
    plt.grid(True, alpha=0.25)
    plt.ylim(-0.05, 1.05)
    plt.show()


def plot_kd_gate_check():
    V = np.linspace(-100, 50, 301)
    mInf = 1 - 1 / (1 + np.exp((V - (-43)) / 8))
    hInf = 1 / (1 + np.exp((V - (-67)) / 7.3))

    plt.figure(figsize=(8, 5))
    plt.plot(V, mInf, label="m∞ activation", linewidth=2)
    plt.plot(V, hInf, label="h∞ inactivation", linewidth=2)
    plt.xlabel("Membrane potential (mV)")
    plt.ylabel("Steady-state value")
    plt.title("Kd Activation and Inactivation Curves")
    plt.legend()
    plt.grid(True, alpha=0.25)
    plt.ylim(-0.05, 1.05)
    plt.show()


plot_nata_gate_check()
plot_kd_gate_check()


## Additional ACT Reference

For ACT-side examples, see the ACT segmentation examples such as `experiments/current_segmentation.ipynb` in the ACT repository.
